In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.getcwd())

In [ ]:
!pip install rasterio retry

In [ ]:
# !pip install geemap
!pip install retry

In [ ]:
import os
import ee
import geemap
import sys
import logging
from multiprocessing import Pool, cpu_count
import requests
import shutil
from retry import retry
import pandas as pd
from itertools import chain
import geopandas as gpd
import ast
from shapely import wkt
from shapely.geometry import mapping

In [ ]:
ee.Authenticate()
ee.Initialize(
        project="ee-sugiscience",
        opt_url="https://earthengine-highvolume.googleapis.com",
)

In [ ]:
#Sentinel 2のパラメータ設定
START_DATE = '2020-04-01'
END_DATE = '2021-04-01'
CLOUD_FILTER = 60
CLD_PRB_THRESH = 50
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 50
TILE_SIZE = 500


In [ ]:
#csvファイルの読み込み
csv_path = '/content/drive/MyDrive/Students/Kanno/oem_labels_only_for_gee.csv'
df = pd.read_csv(csv_path)
rows = df.to_dict('records')

In [ ]:
import re

def sanitize_filename(fname):
    # 許可文字以外をすべて "_"
    fname = re.sub(r'[^a-zA-Z0-9\._,:;\-]', '_', fname)
    # 長さ制限 100文字
    return fname[:100]


In [ ]:
# WKT(well known text)形式の文字列をEarth Engineのジオメトリに変換する関数
def wkt_to_ee_geometry_global(wkt_str):
    geom = wkt.loads(wkt_str)
    return ee.Geometry(mapping(geom), 'EPSG:32634', True, False)

In [ ]:
def get_s2_sr_cld_col(roi, start_date, end_date):
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(roi)
                 .filterDate(start_date, end_date)
                 .sort('system:time_start'))

    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
                        .filterBounds(roi)
                        .filterDate(start_date, end_date)
                        .sort('system:time_start'))

    return ee.ImageCollection(
      ee.Join.saveFirst('s2cloudless').apply(
        primary=s2_sr_col,
        secondary=s2_cloudless_col,
        condition=ee.Filter.equals(leftField='system:index', rightField='system:index')
      )
    )


In [ ]:
def add_cloud_bands(img):
    # 's2cloudless' プロパティから雲確率画像を取得し、'probability' バンドを選択
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')

    # 雲確率が閾値 (CLD_PRB_THRESH) より大きいピクセルを雲 (1) とするマスクを作成
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')

    return img.addBands(ee.Image([cld_prb, is_cloud]))

def add_shadow_bands(img):
    # SCLバンドが6 (水域) でないピクセルを選択 (水域は影と誤認しやすいため除外)
    not_water = img.select('SCL').neq(6)

    # SRデータのスケールファクター (反射率は通常0-1だが、GEEでは整数で格納されるため)
    SR_BAND_SCALE = 1e4
    # 近赤外(B8)の値が閾値 (NIR_DRK_THRESH) より低く、かつ水域でないピクセルを影候補 (dark_pixels) とする
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE).multiply(not_water).rename('dark_pixels')

    # 太陽の方位角から影の方向を計算 (太陽と反対方向)
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')));

    # 雲 ('clouds' バンド) から指定距離 (CLD_PRJ_DIST*10 = km*1000/100m_scale) まで影方向に投影
    # 100mスケールで計算し、投影された領域 (distance > 0) をマスクとして取得
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    # 影候補 (dark_pixels) と 影投影領域 (cld_proj) の積をとり、影 (shadows) を特定
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')

    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

def add_cld_shdw_mask(img):
    # 雲バンドを追加
    img_cloud = add_cloud_bands(img)

    # 影バンドを追加
    img_cloud_shadow = add_shadow_bands(img_cloud)

    # 雲マスクと影マスクを結合 (どちらかが1なら1)
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)

    # モルフォロジー演算でマスクを整形
    # focalMin(2): 小さな穴を埋める (半径2ピクセル)
    # focalMax(BUFFER*2/20): マスクを拡張 (半径 BUFFER m / (20m/pixel) ピクセル)
    # 処理速度のため20m解像度で実行
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))

    # 作成したマスクで画像を更新 (マスク領域は除外されるように .Not() を適用)
    return img_cloud_shadow.updateMask(is_cld_shdw.Not())

In [ ]:
import time

def process_polygon(row):
    # ファイル名（Drive 用）
    filename_drive = sanitize_filename(row['filename'])  # 元のfilename列をベースに
    # タスク名はユニークにするためタイムスタンプを付与
    # description は sanitze_filename を適用し、長さを考慮
    #description_task = f"{filename_drive}_{int(time.time())}"[:100]


    # ROI を WKT から作成
    roi = row['ee_geometry']

    # S2 + S2Cloudless 取得
    s2_col = get_s2_sr_cld_col(roi, START_DATE, END_DATE)
    print(s2_col.size().getInfo())

    # 雲影除去
    s2_masked_col = s2_col.map(add_cld_shdw_mask)

    # 中央値合成 & ROIでクリップ
    img_median = s2_masked_col.median().clip(roi).toFloat()

    return img_median

In [ ]:
def getRequests():
    """Generates a list of work items to be downloaded."""
    return df.index


@retry(tries=10, delay=1, backoff=2)
def getResult(index, regionID):
    """Handle the HTTP requests to download an image."""
    row = df.iloc[regionID]

    # --- ROI を EPSG:4326 として作成 ---
    from shapely import wkt as shapely_wkt
    geom = shapely_wkt.loads(row['geometry_wkt_4326'])
    geojson = geom.__geo_interface__
    roi = ee.Geometry(geojson)  # Earth Engine は lon/lat を期待

    # --- Sentinel-2 の取得 + マスク ---
    s2_col = get_s2_sr_cld_col(roi, START_DATE, END_DATE).map(add_cld_shdw_mask)
    img = s2_col.median()

    """
    # --- 必要バンドを確認 ---
    all_bands = img.bandNames().getInfo()
    exclude_bands = ['B23', 'B24', 'B25']
    keep_bands = [b for b in all_bands if b not in exclude_bands]

    img = img.select(keep_bands)

    """

    required_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A','B11', 'B12']
    available_bands = img.bandNames().getInfo()
    missing_bands = [b for b in required_bands if b not in available_bands]
    if missing_bands:
        print(f"Skipping {index}: missing bands {missing_bands}")
        return

    # --- 必要バンドだけを選択 ---
    img = img.select(required_bands)

    # --- Clip のみ（再投影はしない） ---
    img_clipped = img.clip(roi).toFloat()

    # --- ダウンロード ---
    url = img_clipped.getDownloadURL({
        "region": roi,
        "scale": 10,
        "format": "GEO_TIFF"
    })

    r = requests.get(url)
    r.raise_for_status()

    # --- 保存 ---
    base_name = os.path.basename(str(row["filename"]))   # 例: "aachen/labels/aachen_1.tif"
    safe_name = sanitize_filename(os.path.splitext(base_name)[0])
    filename = f"{export_path}/{safe_name}.tif"

    with open(filename, "wb") as out_file:
        out_file.write(r.content)

    print("Done:", index)


if __name__ == "__main__":
    logging.basicConfig()
    items = getRequests()
    print("Total grids:", len(items))

    export_path = '/content/drive/MyDrive/Students/Kanno/s2_allbands_rej'
    os.makedirs(export_path, exist_ok=True)

    pool = Pool(26)
    pool.starmap(getResult, enumerate(items))
    pool.close()
    pool.join()
